# Medi LLM — Respiratory Virus Q&A Assistant

A fully grounded RAG assistant for respiratory viral illness questions.
Answers are generated **only** from the medical knowledge base — the LLM cannot hallucinate facts outside the indexed documents.

| Stage | Module |
|---|---|
| Load + Chunk | `document_loader.py` / `text_chunker.py` |
| Embed + Index | `embeddings.py` / `vector_store.py` |
| Retrieve | `retriever.py` — ANN over-fetch |
| Rerank | `reranker.py` — cross-encoder (`ms-marco-MiniLM-L-6-v2`) |
| Answer | `llm.py` — Claude with grounding guardrails |

In [ ]:
from pathlib import Path
import sys

# Works whether the kernel CWD is medi_llm/ or the repo root
_cwd = Path.cwd()
_medi_llm_path = _cwd if (_cwd / "knowledge_base").exists() else _cwd / "medi_llm"
if str(_medi_llm_path) not in sys.path:
    sys.path.insert(0, str(_medi_llm_path))

import gradio as gr
from dotenv import load_dotenv

from config import DEFAULT_SETTINGS
from pipeline import build_knowledge_base
from vector_store import get_collection_count
from llm import stream_answer_with_sources, NO_INFO_MESSAGE
from schemas import RetrievalResult

load_dotenv(override=True)
print("Imports OK")

: 

In [2]:
# ── Knowledge-base lazy initialisation ────────────────────────────────────────

_kb_ready = False

def ensure_kb_ready() -> str:
    """Build the ChromaDB index if it has not been built yet, then return a status string."""
    global _kb_ready
    if _kb_ready:
        return f"✅ Knowledge base ready ({get_collection_count()} chunks indexed)"

    count = get_collection_count()
    if count > 0:
        _kb_ready = True
        return f"✅ Knowledge base ready ({count} chunks indexed)"

    artifacts = build_knowledge_base(
        knowledge_base_path=DEFAULT_SETTINGS.knowledge_base_path,
        vector_db_path=DEFAULT_SETTINGS.vector_db_path,
        collection_name=DEFAULT_SETTINGS.collection_name,
        embedding_model_name=DEFAULT_SETTINGS.embedding_model_name,
        chunk_size=DEFAULT_SETTINGS.chunk_size,
        chunk_overlap=DEFAULT_SETTINGS.chunk_overlap,
        reset_existing=False,
    )
    _kb_ready = True
    return f"✅ Knowledge base built ({artifacts.indexed_chunk_count} chunks indexed)"


# ── Gradio helper: source citations panel ─────────────────────────────────────

_SCORE_COLORS = [
    (5.0,  "#16a34a"),   # green  — highly relevant
    (2.0,  "#2563eb"),   # blue   — relevant
    (0.0,  "#d97706"),   # amber  — marginally relevant
    (-999, "#94a3b8"),   # grey   — low / unknown
]

def _score_color(score) -> str:
    """Return a hex colour based on the cross-encoder rerank score."""
    if score is None:
        return "#94a3b8"
    for threshold, color in _SCORE_COLORS:
        if score >= threshold:
            return color
    return "#94a3b8"


def build_sources_html(sources: list) -> str:
    """Render retrieved chunks as an HTML citation panel."""
    if not sources:
        return (
            "<p style='color:#94a3b8;font-size:0.85em;font-family:system-ui;"
            "padding:12px 4px'>No sources retrieved.</p>"
        )

    cards = []
    for i, r in enumerate(sources, start=1):
        source  = r.metadata.get("source", "unknown")
        section = r.metadata.get("section_heading", "")
        score   = r.rerank_score
        color   = _score_color(score)
        score_label = f"{score:.2f}" if score is not None else "n/a"
        preview = r.text[:220].replace("<", "&lt;").replace(">", "&gt;")
        if len(r.text) > 220:
            preview += "…"

        cards.append(f"""
<div style="background:#f8fafc;border:1px solid #e2e8f0;border-left:3px solid {color};
            border-radius:8px;padding:10px 14px;margin-bottom:8px;font-family:system-ui">
  <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:4px">
    <span style="font-size:0.73em;color:#64748b;font-weight:600">[{i}] {source}</span>
    <span style="background:{color};color:#fff;padding:1px 7px;border-radius:10px;
                 font-size:0.7em;font-weight:700">score {score_label}</span>
  </div>
  <div style="font-size:0.75em;color:#475569;margin-bottom:5px;font-style:italic">{section}</div>
  <div style="font-size:0.78em;color:#334155;line-height:1.55">{preview}</div>
</div>""")

    header = (
        f"<div style='font-family:system-ui;font-weight:700;font-size:0.88em;"
        f"color:#1e293b;margin-bottom:8px'>📚 {len(sources)} source chunk"
        f"{'s' if len(sources) != 1 else ''}</div>"
    )
    return header + "".join(cards)


# ── Gradio streaming chat handler ─────────────────────────────────────────────

def respond(message: str, history: list):
    """Stream a grounded answer while progressively updating the chat + sources panel.

    Yields (updated_history, sources_html, cleared_input) at each streaming step.
    """
    message = message.strip()
    if not message:
        yield history, "", message
        return

    try:
        ensure_kb_ready()
    except Exception as exc:
        history = history + [
            {"role": "user",      "content": message},
            {"role": "assistant", "content": f"⚠️ Knowledge base initialisation failed: {exc}"},
        ]
        yield history, "", ""
        return

    history = history + [
        {"role": "user",      "content": message},
        {"role": "assistant", "content": ""},
    ]

    sources = []
    try:
        for partial_text, partial_sources in stream_answer_with_sources(message):
            history[-1]["content"] = partial_text
            if partial_sources is not None:
                sources = partial_sources or []
            else:
                # mid-stream: update chat only, leave sources blank while typing
                yield history, "", ""
    except Exception as exc:
        history[-1]["content"] = f"⚠️ Error generating answer: {exc}"
        yield history, "", ""
        return

    # Final yield: complete answer + rendered sources
    yield history, build_sources_html(sources), ""


print("Helpers ready.")

Loaded 10 documents
Created 113 chunks

Sample document metadata:
{'document_type': 'foundation', 'topic': 'viral_pathogenesis_and_transmission', 'audience': 'rag_system', 'source_family': 'CDC, WHO, PMC', 'relative_path': '01_virus_foundations/how_respiratory_viruses_cause_disease.md', 'file_name': 'how_respiratory_viruses_cause_disease.md', 'category': '01_virus_foundations'}

Sample chunk preview:
How Respiratory Viruses Cause Disease > Entry Into The Body

Respiratory viruses usually enter through the nose, mouth, or eyes after exposure to infectious respiratory particles or contaminated secretions. After entry, the virus attaches to cells lining the respiratory tract and starts replicating.


In [3]:
_HEADER_HTML = """
<div style="font-family:system-ui,sans-serif;padding:20px 8px 12px;
            background:linear-gradient(135deg,#0f172a 0%,#1e3a5f 100%);
            border-radius:12px;margin-bottom:4px">
  <div style="display:flex;align-items:center;gap:12px">
    <span style="font-size:2.2em">🫁</span>
    <div>
      <h1 style="margin:0;font-size:1.5em;color:#f1f5f9;font-weight:800;
                 letter-spacing:-0.02em">Medi LLM</h1>
      <p style="margin:2px 0 0;color:#94a3b8;font-size:0.85em">
        Respiratory virus Q&amp;A — grounded in CDC &amp; WHO knowledge base
      </p>
    </div>
  </div>
  <p style="margin:12px 0 0;color:#cbd5e1;font-size:0.78em;line-height:1.6">
    Ask about <b style="color:#7dd3fc">common cold</b>,
    <b style="color:#86efac">influenza</b>,
    <b style="color:#fca5a5">COVID-19</b>, or
    <b style="color:#fdba74">RSV</b> —
    symptoms, treatments, prevention, and how they differ.
    Answers are drawn exclusively from the indexed knowledge base.
  </p>
</div>
"""

_EXAMPLE_QUESTIONS = [
    "What are the main symptoms of influenza?",
    "How do COVID-19 and the common cold differ?",
    "What antiviral medicines are used for influenza?",
    "Who is most at risk from RSV?",
    "Which respiratory infections have vaccines available?",
    "How does RSV cause bronchiolitis in infants?",
]

_NO_SOURCES_PLACEHOLDER = (
    "<p style='color:#94a3b8;font-size:0.85em;font-family:system-ui;padding:8px 4px'>"
    "Ask a question to see the knowledge-base chunks used to generate the answer.</p>"
)

with gr.Blocks(
    title="Medi LLM — Respiratory Virus Assistant",
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate",
        neutral_hue="slate",
    ),
    css="""
    .gradio-container { max-width: 1200px !important; }
    footer { display: none !important; }
    """,
) as demo:

    # ── Header ────────────────────────────────────────────────────────────────
    gr.HTML(_HEADER_HTML)

    # ── Status bar ────────────────────────────────────────────────────────────
    status_bar = gr.Textbox(
        value="⏳ Click 'Initialise Knowledge Base' to load the index…",
        label="",
        interactive=False,
        show_label=False,
        container=False,
        elem_id="status_bar",
    )
    init_btn = gr.Button("⚡ Initialise Knowledge Base", variant="secondary", size="sm")

    gr.HTML("<hr style='border:none;border-top:1px solid #e2e8f0;margin:8px 0'>")

    # ── Main layout ───────────────────────────────────────────────────────────
    with gr.Row(equal_height=True):

        # Left: chat column
        with gr.Column(scale=3, min_width=420):
            chatbot = gr.Chatbot(
                value=[],
                type="messages",
                render_markdown=True,
                height=500,
                bubble_full_width=False,
                show_label=False,
                avatar_images=(None, None),
                placeholder=(
                    "<div style='text-align:center;padding:40px 20px;"
                    "color:#94a3b8;font-family:system-ui'>"
                    "<div style='font-size:2.5em;margin-bottom:8px'>🫁</div>"
                    "<p style='font-size:0.9em'>Ask a question about respiratory viruses.</p>"
                    "</div>"
                ),
            )

            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="e.g. What are the symptoms of influenza?",
                    show_label=False,
                    lines=1,
                    scale=9,
                    container=False,
                    autofocus=True,
                )
                send_btn = gr.Button("Send →", variant="primary", scale=2, min_width=90)

            gr.Examples(
                examples=_EXAMPLE_QUESTIONS,
                inputs=msg_input,
                label="Example questions",
                examples_per_page=3,
            )

            clear_btn = gr.Button("🗑 Clear conversation", variant="secondary", size="sm")

        # Right: sources column
        with gr.Column(scale=2, min_width=280):
            gr.Markdown("### 📚 Retrieved Sources")
            gr.Markdown(
                "<small style='color:#64748b'>Knowledge-base chunks used to generate "
                "the answer, ordered by cross-encoder rerank score.</small>",
            )
            sources_panel = gr.HTML(value=_NO_SOURCES_PLACEHOLDER)

    # ── Event wiring ──────────────────────────────────────────────────────────
    _submit_inputs  = [msg_input, chatbot]
    _submit_outputs = [chatbot, sources_panel, msg_input]

    send_btn.click(fn=respond, inputs=_submit_inputs, outputs=_submit_outputs)
    msg_input.submit(fn=respond, inputs=_submit_inputs, outputs=_submit_outputs)

    clear_btn.click(
        fn=lambda: ([], _NO_SOURCES_PLACEHOLDER),
        outputs=[chatbot, sources_panel],
    )

    init_btn.click(fn=ensure_kb_ready, outputs=status_bar)
    demo.load(fn=ensure_kb_ready, outputs=status_bar)

print("Gradio app defined.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 113 chunks
Collection count after indexing: 113


In [2]:
demo.launch(share=False)

Query : How do influenza and the common cold differ in symptoms and severity?
Top-K : 5  |  Candidates fetched for reranking: 15

STAGE 1 — Plain ANN retrieval (vector distance only)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.




Result 1
Source   : 02_disease_profiles/influenza.md
Section  : Influenza > How It Differs From Other Respiratory Viral Illnesses
Distance : 0.4690  |  Rerank score : n/a

Influenza > How It Differs From Other Respiratory Viral Illnesses

- Flu often begins more abruptly than the common cold.
- Fever, chills, muscle aches, and exhaustion are more prominent than in a routine cold.
- It may overlap with COVID-19, but symptoms alone cannot reliably distinguish the two.
- Flu is more likely than the common cold to cause complications such as pneumonia or worsening of chronic disease.

--------------------------------------------------------------------------------

Result 2
Source   : 02_disease_profiles/common_cold.md
Section  : Common Cold > How It Differs From Other Respiratory Viral Illnesses
Distance : 0.5409  |  Rerank score : n/a

Common Cold > How It Differs From Other Respiratory Viral Illnesses

- It is usually milder than influenza.
- It is less likely than flu to cause sudden

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]



Result 1
Source   : 02_disease_profiles/influenza.md
Section  : Influenza > How It Differs From Other Respiratory Viral Illnesses
Distance : 0.4690  |  Rerank score : 7.1795

Influenza > How It Differs From Other Respiratory Viral Illnesses

- Flu often begins more abruptly than the common cold.
- Fever, chills, muscle aches, and exhaustion are more prominent than in a routine cold.
- It may overlap with COVID-19, but symptoms alone cannot reliably distinguish the two.
- Flu is more likely than the common cold to cause complications such as pneumonia or worsening of chronic disease.

--------------------------------------------------------------------------------

Result 2
Source   : 02_disease_profiles/common_cold.md
Section  : Common Cold > How It Differs From Other Respiratory Viral Illnesses
Distance : 0.5409  |  Rerank score : 6.6751

Common Cold > How It Differs From Other Respiratory Viral Illnesses

- It is usually milder than influenza.
- It is less likely than flu to cause 



Result 1
Source   : 02_disease_profiles/covid_19.md
Section  : COVID-19 > Medications > Antiviral Options Mentioned By CDC
Distance : 0.5188  |  Rerank score : 7.1537

COVID-19 > Medications > Antiviral Options Mentioned By CDC

- nirmatrelvir plus ritonavir
- remdesivir
- molnupiravir

These medications are most useful when started early in illness and according to eligibility criteria.

--------------------------------------------------------------------------------

Result 2
Source   : 03_prevention_treatment_medications/treatments_and_medications_by_infection.md
Section  : Treatments And Medications By Infection > 3. COVID-19 > Antiviral Medications Mentioned By CDC
Distance : 0.4640  |  Rerank score : 7.0041

Treatments And Medications By Infection > 3. COVID-19 > Antiviral Medications Mentioned By CDC

- nirmatrelvir plus ritonavir
- remdesivir
- molnupiravir

--------------------------------------------------------------------------------

Result 3
Source   : 03_prevention_tre



Result 1
Source   : 02_disease_profiles/rsv.md
Section  : Respiratory Syncytial Virus (RSV) > How It Differs From Other Respiratory Viral Illnesses
Distance : 0.5412  |  Rerank score : 6.3056

Respiratory Syncytial Virus (RSV) > How It Differs From Other Respiratory Viral Illnesses

- RSV is especially important in infants and young children because it can cause bronchiolitis and pneumonia.
- It is also clinically important in older adults and patients with chronic cardiopulmonary disease.
- Unlike influenza and COVID-19, routine outpatient treatment is usually supportive rather than antiviral.

--------------------------------------------------------------------------------

Result 2
Source   : 02_disease_profiles/respiratory_virus_differences.md
Section  : Differences Between Common Cold, Influenza, COVID-19, And RSV > Quick Comparison > RSV
Distance : 0.8899  |  Rerank score : 2.3919

Differences Between Common Cold, Influenza, COVID-19, And RSV > Quick Comparison > RSV

- common 



Result 1
Source   : 03_prevention_treatment_medications/prevention_and_infection_control.md
Section  : Prevention And Infection Control For Common Respiratory Viral Infections > Vaccination And Immunization > RSV
Distance : 0.7747  |  Rerank score : 1.3950

Prevention And Infection Control For Common Respiratory Viral Infections > Vaccination And Immunization > RSV

- prevention now includes immunization approaches for selected groups
- these may include maternal vaccination, certain adult vaccines, or long-acting monoclonal antibody products for infants depending on setting and eligibility

--------------------------------------------------------------------------------

Result 2
Source   : 03_prevention_treatment_medications/prevention_and_infection_control.md
Section  : Prevention And Infection Control For Common Respiratory Viral Infections > Vaccination And Immunization > Influenza
Distance : 0.7916  |  Rerank score : 1.2920

Prevention And Infection Control For Common Respirato

Query: What causes fever and muscle aches in respiratory infections?

Rerank   ANN  Rerank score  ANN dist  Section
---------------------------------------------
     1     2        5.7313    0.8149  Definitions And Types Of Respiratory Viruses > Major Re
     2     1        4.9711    0.4995  How Respiratory Viruses Cause Disease > Common Mechanis
     3     3        3.7581    0.8194  Common Cold > How It Differs From Other Respiratory Vir
     4     9        1.2287    0.9367  Differences Between Common Cold, Influenza, COVID-19, A
     5    11       -2.8915    0.9381  How Respiratory Viruses Cause Disease > Common Mechanis


Question : What antiviral medicines are used for influenza?

Answer:
According to the CDC, the following antiviral medications are used to treat influenza:

- **oseltamivir**
- **zanamivir**
- **peramivir**
- **baloxavir**

These antivirals work best when started early, especially within about 48 hours of symptom onset. Early antiviral treatment is particularly important for high-risk patients and those with severe disease. (Source: 02_disease_profiles/influenza.md)

Sources used (5):
  - 02_disease_profiles/influenza.md  |  score: 9.293567
  - 03_prevention_treatment_medications/treatments_and_medications_by_infection.md  |  score: 8.310699
  - 02_disease_profiles/influenza.md  |  score: 3.58196
  - 03_prevention_treatment_medications/treatments_and_medications_by_infection.md  |  score: 3.46546
  - 03_prevention_treatment_medications/treatments_and_medications_by_infection.md  |  score: 3.449524


COVID-19 is caused specifically by the virus **SARS-CoV-2** (Source: 02_disease_profiles/covid_19.md). 

This virus is different from influenza viruses, rhinoviruses, and RSV, which cause other respiratory illnesses.
